# Step 0: Base Model 기본 성능 테스트
Llama-3.2-3B로 우리 문제 15개를 풀어보고 기본 정확도 확인.
이게 너무 낮으면 8B로 변경 또는 문제 난이도 조정.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft torch
!pip install -q huggingface_hub

In [ ]:
# HuggingFace 로그인 (Llama 접근 필요)
from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")  # HF 토큰 입력

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time, json, re

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# 모델 로드 (4bit 양자화)
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True,
)
print("Model loaded.")

In [ ]:
def generate(system, user, max_new_tokens=512):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - t0
    response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    gen_tokens = out.shape[1] - input_ids.shape[1]
    return response, gen_tokens, elapsed

In [ ]:
# 15문제 (KI-1A와 동일: 대규모 데이터셋 정보 필터링)
PROBLEMS = [
    {
        "id": 1, "domain": "Corporate Finance",
        "data": "Revenue=$12M, COGS=$7.2M, SGA=$1.8M, RnD=$600K, Interest=$400K, Tax_Rate=28%, Depreciation=$500K, Amortization=$200K, Total_Assets=$25M, Current_Assets=$8M, Fixed_Assets=$15M, Intangibles=$2M, Current_Liabilities=$5M, LongTerm_Debt=$8M, Equity=$12M, Shares_Outstanding=2M, Dividends=$600K, CapEx=$1.5M, Working_Capital_Change=$300K, Cash=$1.5M",
        "b_task": "Compute Debt-to-Equity Ratio = LongTerm_Debt / Equity",
        "b_needs": "LongTerm_Debt=$8M, Equity=$12M",
        "answer": 0.667
    },
    {
        "id": 2, "domain": "Manufacturing",
        "data": "Production_Rate=500/hr, Defect_Rate=2.5%, Rework_Rate=1.5%, Scrap_Rate=1.0%, Material_Cost=$15/unit, Labor_Cost=$8/unit, Overhead=$3/unit, Energy=$2/unit, Maintenance=$50K/month, Downtime=5%, Shift_Hours=8, Shifts_Per_Day=3, Workers=120, Machines=25, Capacity_Utilization=85%, Safety_Incidents=3/yr, Waste_Rate=4%, Quality_Score=94",
        "b_task": "Good_Units_Per_Day = Production_Rate * (1-Defect_Rate) * Shift_Hours * Shifts_Per_Day * (1-Downtime)",
        "b_needs": "Production_Rate=500, Defect_Rate=2.5%, Shift_Hours=8, Shifts=3, Downtime=5%",
        "answer": 11115
    },
    {
        "id": 3, "domain": "Healthcare",
        "data": "Beds=500, Occupancy=78%, Avg_Stay=4.2days, Admissions=3400/month, ER_Visits=8500/month, Surgery=600/month, Outpatient=12000/month, Staff_Doctors=200, Nurses=600, Admin=150, Budget=$50M/month, Revenue=$55M/month, Insurance_Claims=$45M, Patient_Satisfaction=4.2/5, Readmission_Rate=8%, Mortality_Rate=1.2%, Infection_Rate=0.5%, Wait_Time_ER=45min",
        "b_task": "Bed_Turnover_Rate = Admissions / Beds",
        "b_needs": "Admissions=3400, Beds=500",
        "answer": 6.8
    },
    {
        "id": 4, "domain": "Investment",
        "data": "Stock_A_Return=12%,Weight=30%,Std=25%; Stock_B_Return=8%,Weight=25%,Std=15%; Bond_C_Return=4%,Weight=20%,Std=5%; REIT_D_Return=9%,Weight=15%,Std=18%; Cash_Return=2%,Weight=10%,Std=0%; Corr_AB=0.6, Corr_AC=-0.2, Corr_AD=0.4, Risk_Free=3%, Benchmark=10%",
        "b_task": "Portfolio_Expected_Return = weighted sum of returns",
        "b_needs": "All returns and weights",
        "answer": 7.95
    },
    {
        "id": 5, "domain": "Demographics",
        "data": "Population=850000, Area=350km2, Growth_Rate=1.8%, Median_Age=34, Under18=22%, Over65=14%, Median_Income=$52000, Unemployment=5.5%, Poverty_Rate=12%, College_Educated=38%, Homeownership=55%, Avg_Rent=$1200, Crime_Rate=4.2/1000, Schools=120, Hospitals=8, Parks=45, Transit_Ridership=150000/day",
        "b_task": "People_Over_65 = Population * Over65_Pct",
        "b_needs": "Population=850000, Over65=14%",
        "answer": 119000
    },
    {
        "id": 6, "domain": "Logistics",
        "data": "Warehouses=3, Trucks=45, Avg_Delivery_Time=2.5days, Orders_Per_Day=850, Return_Rate=4%, Fuel_Cost=$0.35/km, Avg_Distance=120km, Staff=200, Capacity=1000orders/day, Peak_Multiplier=1.5, Damage_Rate=0.5%, Insurance=$200K/yr, Tech_Budget=$500K, Customer_Satisfaction=4.1/5, On_Time_Rate=94%, Avg_Order_Value=$22",
        "b_task": "Annual_Throughput = Orders_Per_Day * 365 * (1 - Return_Rate) * (1 - Damage_Rate) / Warehouses * Warehouses",
        "b_needs": "Orders_Per_Day=850, Return_Rate=4%, Damage_Rate=0.5%",
        "answer": 18700
    },
    {
        "id": 7, "domain": "Energy",
        "data": "Capacity=500MW, Utilization=72%, Fuel_Cost=$3.2/MWh, Maintenance=$1.5M/yr, Staff=150, Avg_Price=$45/MWh, Peak_Price=$120/MWh, Off_Peak=$25/MWh, Emissions=0.5tCO2/MWh, Renewable_Pct=15%, Grid_Loss=6%, Downtime=8%, Capital_Cost=$800M, Debt_Service=$50M/yr, Regulatory_Fee=$5M/yr, Customer_Count=200000, Avg_Bill=$150/month",
        "b_task": "Cost_Per_MWh = (Fuel_Cost + Maintenance_Per_MWh)",
        "b_needs": "Fuel_Cost=$3.2/MWh, Maintenance=$1.5M/yr, Annual_Generation",
        "answer": 19.05
    },
    {
        "id": 8, "domain": "Agriculture",
        "data": "Farm_Size=2000acres, Crop_A=Corn(800acres,180bu/acre), Crop_B=Soybean(600acres,50bu/acre), Crop_C=Wheat(400acres,60bu/acre), Fallow=200acres, Water_Usage=500K_gal/day, Fertilizer=$200/acre, Pesticide=$80/acre, Labor=50workers, Equipment_Value=$2M, Soil_pH=6.5, Rainfall=35in/yr, Irrigation=40%, Organic_Pct=10%, Subsidy=$50K, Insurance=$100K",
        "b_task": "Total_Yield_Bushels = sum of (acres * yield) for each crop",
        "b_needs": "Corn 800ac*180bu, Soybean 600*50, Wheat 400*60",
        "answer": 835200
    },
    {
        "id": 9, "domain": "Retail",
        "data": "Stores=120, Employees=4500, Revenue=$4.2B, COGS=$2.8B, Online_Pct=25%, Avg_Transaction=$35, Footfall=2M/week, Conversion=30%, Return_Rate=8%, Inventory_Turnover=6, Marketing=$200M, Loyalty_Members=5M, Avg_Basket=3.2items, Shrinkage=1.5%, Rent=$300M, Salary=$800M, Distribution_Centers=5",
        "b_task": "Revenue_Per_Employee = Revenue / Employees",
        "b_needs": "Revenue=$4.2B, Employees=4500",
        "answer": 933333
    },
    {
        "id": 10, "domain": "University",
        "data": "Students=22000, Faculty=700, Staff=1200, Tuition=$35000, Acceptance_Rate=32%, Graduation_Rate=88%, Endowment=$2.5B, Research_Funding=$300M, Dorms=8000beds, Library_Books=3M, Alumni=150000, Avg_Class_Size=28, Student_Faculty_Ratio=?, International=18%, Graduate=35%, Departments=45, Patents=120/yr, Ranking=Top50",
        "b_task": "Student_Faculty_Ratio = Students / (Faculty + 0.5*TAs)",
        "b_needs": "Students=22000, Faculty=700",
        "answer": 15.91
    },
    {
        "id": 11, "domain": "Sports",
        "data": "Players=15, Avg_Age=27, Salary_Cap=$140M, Total_Salary=$132M, Wins=48, Losses=34, Home_Record=28-13, Away=20-21, Avg_Points=95.14, Opp_Points=92.8, FG_Pct=46.2%, 3PT_Pct=36.5%, FT_Pct=78.1%, Rebounds=44.2, Assists=25.1, Turnovers=13.5, Attendance=18500, Revenue=$350M, Merchandise=$45M",
        "b_task": "Avg_Points_Per_Game = given directly in data",
        "b_needs": "Avg_Points=95.14",
        "answer": 95.14
    },
    {
        "id": 12, "domain": "Weather",
        "data": "Jan_High=35, Feb=38, Mar=48, Apr=58, May=68, Jun=78, Jul=85, Aug=83, Sep=75, Oct=63, Nov=50, Dec=39, Avg_Precip=3.5in/month, Snowfall=25in/yr, Sunny_Days=200, Humidity=65%, Wind_Avg=12mph, UV_Index=6, Record_High=105, Record_Low=-15, Growing_Season=180days",
        "b_task": "Average_High_Temperature = mean of monthly highs",
        "b_needs": "All 12 monthly highs",
        "answer": 60.0
    },
    {
        "id": 13, "domain": "Telecom",
        "data": "Subscribers=25M, ARPU=$55, Churn=1.8%/month, Towers=50000, Coverage=98%, 5G_Coverage=45%, Data_Usage=15GB/user/month, Voice_Min=300/user, SMS=50/user, Spectrum_Cost=$5B, CapEx=$3B/yr, OpEx=$8B/yr, Revenue=$16.5B, EBITDA=$5.5B, Debt=$12B, Employees=30000, NPS=35",
        "b_task": "Total_Monthly_Data = Subscribers * Data_Usage",
        "b_needs": "Subscribers=25M, Data_Usage=15GB",
        "answer": 375000000
    },
    {
        "id": 14, "domain": "Airline",
        "data": "Fleet=250, Routes=300, Passengers=80M/yr, Load_Factor=84%, Avg_Fare=$220, Ancillary=$35/pax, Fuel_Cost=$8B, Labor=$5B, Maintenance=$2B, Revenue=$22B, Operating_Margin=8%, Hubs=5, Destinations=150, On_Time=82%, Baggage_Loss=2.5/1000, Loyalty_Members=40M, Avg_Flight_Distance=1200mi",
        "b_task": "Revenue_Per_Passenger = (Revenue / Passengers)",
        "b_needs": "Revenue=$22B, Passengers=80M",
        "answer": 275
    },
    {
        "id": 15, "domain": "Real Estate",
        "data": "Units=500, Avg_Size=900sqft, Occupancy=92%, Avg_Rent=$1800/month, Operating_Expenses=$4M/yr, Property_Value=$160M, Cap_Rate=5.5%, Mortgage=$100M, Interest_Rate=4.5%, Property_Tax=$2M, Insurance=$500K, HOA=$200/unit, Vacancy_Loss=8%, Turnover=20%/yr, Maintenance=$1.5M, Land_Value=$40M",
        "b_task": "Price_Per_SqFt = Property_Value / (Units * Avg_Size)",
        "b_needs": "Property_Value=$160M, Units=500, Avg_Size=900sqft",
        "answer": 355.56
    },
]

In [ ]:
# Test 1: Agent A (base model) — No Context, 자유 요약
print("=" * 60)
print("TEST: Base Model as Agent A (No Context)")
print("=" * 60)

a_system = "You are a data analyst. Given the following dataset, write a summary of the key findings. Be comprehensive."

results_a = []
for p in PROBLEMS:
    resp, tokens, elapsed = generate(a_system, f"Dataset:\n{p['data']}")
    results_a.append({"id": p["id"], "domain": p["domain"], "response": resp, "tokens": tokens, "time": elapsed})
    print(f"  P{p['id']:2d} ({p['domain']:20s}): {tokens:4d} tokens, {elapsed:.1f}s")
    print(f"       First 100 chars: {resp[:100]}")

avg_tokens = sum(r["tokens"] for r in results_a) / len(results_a)
total_time = sum(r["time"] for r in results_a)
print(f"\nAvg tokens: {avg_tokens:.0f}, Total time: {total_time:.1f}s")

In [ ]:
# Test 2: Agent B (base model) — A의 출력을 받아서 계산
print("=" * 60)
print("TEST: Base Model as Agent B (receives A's summary)")
print("=" * 60)

b_system = "Using ONLY the information in the data summary below, compute the requested value. If the needed data is not in the summary, output -999. Output ONLY the final numeric answer."

correct = 0
for i, p in enumerate(PROBLEMS):
    a_msg = results_a[i]["response"]
    b_input = f"Data summary:\n{a_msg}\n\nTask: {p['b_task']}"
    resp, tokens, elapsed = generate(b_system, b_input, max_new_tokens=100)
    
    # Parse number
    nums = re.findall(r'-?[\d,]+\.?\d*', resp.replace(',', ''))
    b_answer = float(nums[0]) if nums else -999
    
    expected = p["answer"]
    is_correct = abs(b_answer - expected) / max(abs(expected), 1e-9) < 0.10
    if is_correct: correct += 1
    
    status = "OK" if is_correct else "FAIL"
    print(f"  P{p['id']:2d}: B={b_answer}, expected={expected}, {status} ({elapsed:.1f}s)")

print(f"\nAccuracy: {correct}/{len(PROBLEMS)} ({correct/len(PROBLEMS)*100:.1f}%)")

In [ ]:
# Test 3: Agent A — Mutual (B가 필요한 데이터 명시)
print("=" * 60)
print("TEST: Base Model as Agent A (Mutual — B requests specific data)")
print("=" * 60)

a_system_mutual = "The recipient requested specific data. Send ONLY what they asked for, as key=value pairs. Nothing else."

results_mutual = []
correct_mutual = 0

for p in PROBLEMS:
    # A receives B's request
    a_input = f"Dataset:\n{p['data']}\n\nRecipient request: I need {p['b_needs']} to compute {p['b_task']}"
    a_resp, a_tokens, _ = generate(a_system_mutual, a_input, max_new_tokens=100)
    
    # B receives A's compact message
    b_input = f"Data summary:\n{a_resp}\n\nTask: {p['b_task']}"
    b_resp, _, elapsed = generate(b_system, b_input, max_new_tokens=100)
    
    nums = re.findall(r'-?[\d,]+\.?\d*', b_resp.replace(',', ''))
    b_answer = float(nums[0]) if nums else -999
    expected = p["answer"]
    is_correct = abs(b_answer - expected) / max(abs(expected), 1e-9) < 0.10
    if is_correct: correct_mutual += 1
    
    status = "OK" if is_correct else "FAIL"
    print(f"  P{p['id']:2d}: A sent {a_tokens} tokens, B={b_answer}, expected={expected}, {status}")
    results_mutual.append({"id": p["id"], "a_tokens": a_tokens, "b_answer": b_answer, "correct": is_correct})

avg_mut = sum(r["a_tokens"] for r in results_mutual) / len(results_mutual)
print(f"\nMutual: Avg A tokens={avg_mut:.0f}, Accuracy={correct_mutual}/{len(PROBLEMS)} ({correct_mutual/len(PROBLEMS)*100:.1f}%)")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("STEP 0 SUMMARY")
print("=" * 60)
print(f"Model: {MODEL_ID}")
print(f"\nNo Context: Avg A tokens={avg_tokens:.0f}, B Accuracy={correct}/{len(PROBLEMS)}")
print(f"Mutual:     Avg A tokens={avg_mut:.0f}, B Accuracy={correct_mutual}/{len(PROBLEMS)}")
print(f"\nToken reduction: {(1 - avg_mut/avg_tokens)*100:.1f}%")
print(f"\n→ 이 정확도가 너무 낮으면 (< 30%) 8B 모델로 변경 필요")
print(f"→ 이 정확도가 괜찮으면 (> 50%) LoRA 학습 진행 가능")